# Python Overview — PT Kirimin
Kita membuat transformasi data yang dapat diulang dan menghasilkan quality report.

> Diverifikasi: Python v3.14.7 — sumber: https://docs.python.org/3/ — tanggal cek: 2026-09-18
> Diverifikasi: pandas v3.0.3 — sumber: https://pandas.pydata.org/docs/whatsnew/v3.0.3.html — tanggal cek: 2026-09-18

In [ ]:
import pandas as pd
from dataclasses import dataclass

raw = pd.DataFrame({
    'order_id': ['O-101', 'O-102', 'O-102', None],
    'origin_city': [' jakarta ', 'JKT', 'JKT', 'Bandung'],
    'order_value_idr': ['250000', 'bad', '450000', '900000'],
})
raw

In [ ]:
@dataclass
class QualityReport:
    input_rows: int
    output_rows: int
    invalid_order_values: int
    duplicate_order_ids: int

def transform_orders(frame: pd.DataFrame) -> tuple[pd.DataFrame, QualityReport]:
    required = {'order_id', 'origin_city', 'order_value_idr'}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'missing required columns: {sorted(missing)}')
    result = frame.copy()
    result['origin_city'] = (result['origin_city'].astype('string').str.strip().str.lower()
                             .replace({'jkt': 'jakarta'}).str.title())
    result['order_value_idr'] = pd.to_numeric(result['order_value_idr'], errors='coerce')
    invalid = int(result['order_value_idr'].isna().sum())
    duplicates = int(result['order_id'].duplicated(keep=False).sum())
    result = result.dropna(subset=['order_id']).drop_duplicates('order_id', keep='last')
    report = QualityReport(len(frame), len(result), invalid, duplicates)
    return result, report

cleaned, report = transform_orders(raw)
cleaned, report

## Mini-exercise
1. Tambahkan aturan bahwa `order_value_idr` negatif masuk ke kolom error.
2. Ubah fungsi agar mengembalikan daftar `order_id` yang ditolak.
# TODO: tulis percobaan Anda di bawah cell ini.

## Takeaway
Fungsi pipeline sebaiknya deterministik, eksplisit terhadap kontrak, dan mengeluarkan quality signal selain DataFrame.